<h3 style="
    text-align:center;
    background-color:#5AA647;
    color:white;
    padding:8px;
    border-radius:6px;">
    Hybrid Recommendations
</h3>

In [18]:
from IPython.display import display, HTML

display(HTML("""
<div style="font-family:Arial; width:750px; margin:auto;">

<h3 style="
    text-align:center;
    background-color:#5AA647;
    color:white;
    padding:10px;
    border-radius:8px;">
Hybrid Recommendation Logic
</h3>

<table style="
    width:100%;
    border-collapse: collapse;
    margin-top:10px;
    text-align:center;
">

<tr style="background-color:#DDE5DD;">
    <th style="padding:10px; border:1px solid #ccc;">Type</th>
    <th style="padding:10px; border:1px solid #ccc;">Input</th>
    <th style="padding:10px; border:1px solid #ccc;">Output</th>
    <th style="padding:10px; border:1px solid #ccc;">Logic</th>
</tr>

<tr>
    <td style="padding:10px; border:1px solid #ccc;"><b>User-Based</b></td>
    <td style="padding:10px; border:1px solid #ccc;">User</td>
    <td style="padding:10px; border:1px solid #ccc;">Movies</td>
    <td style="padding:10px; border:1px solid #ccc;">
        Similar users → their movies
    </td>
</tr>

<tr style="background-color:#f9f9f9;">
    <td style="padding:10px; border:1px solid #ccc;"><b>Item-Based</b></td>
    <td style="padding:10px; border:1px solid #ccc;">Movie</td>
    <td style="padding:10px; border:1px solid #ccc;">Users</td>
    <td style="padding:10px; border:1px solid #ccc;">
        Similar movies → new users
    </td>
</tr>

<tr>
    <td style="padding:10px; border:1px solid #ccc;"><b>Content-Based</b></td>
    <td style="padding:10px; border:1px solid #ccc;">User</td>
    <td style="padding:10px; border:1px solid #ccc;">Movies</td>
    <td style="padding:10px; border:1px solid #ccc;">
        Similar features → similar movies
    </td>
</tr>

</table>

</div>
"""))

Type,Input,Output,Logic
User-Based,User,Movies,Similar users → their movies
Item-Based,Movie,Users,Similar movies → new users
Content-Based,User,Movies,Similar features → similar movies


#### Load Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances

# Load movie dataset
i_cols = [
    'movie id', 'movie title', 'release date', 'video release date', 'IMDb URL',
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery',
    'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

items = pd.read_csv('ml-100k/u.item', sep='|', names=i_cols, encoding='latin-1')

# Select usable movie features
movie_content = items[['movie id', 'movie title',
    'Action','Adventure','Animation',"Children's",'Comedy','Crime',
    'Documentary','Drama','Fantasy','Film-Noir','Horror',
    'Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western']].copy()

# Load ratings
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols)

# Create user lookup (for display)
user_lookup = pd.DataFrame({
    'user_id': ratings['user_id'].unique()
})
user_lookup['user_name'] = user_lookup['user_id'].apply(lambda x: f"User-{x}")

In [2]:
# Create once (important for production)
data_matrix = ratings.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
).fillna(0)

#### User Based Recommendation

In [3]:
def user_based(ratings, items, user_id, top_n=10):

    # User similarity
    sim = 1 - pairwise_distances(data_matrix, metric='cosine')
    sim_df = pd.DataFrame(sim, index=data_matrix.index, columns=data_matrix.index)

    # Similar users
    similar_users = sim_df[user_id].sort_values(ascending=False)[1:6]

    watched = set(ratings[ratings['user_id'] == user_id]['movie_id'])

    scores = {}

    for u, score in similar_users.items():
        movies = ratings[ratings['user_id'] == u]['movie_id']
        for m in movies:
            if m not in watched:
                scores[int(m)] = scores.get(int(m), 0) + score

    # Get movie titles
    top_movies = sorted(scores, key=scores.get, reverse=True)[:top_n]

    return items[items['movie id'].isin(top_movies)]['movie title'].tolist()

#### Item Based Recommendation

In [4]:
def item_based_users(ratings, user_lookup, movie_id, top_n=10):

    # Item similarity
    item_sim = 1 - pairwise_distances(data_matrix.T, metric='cosine')

    item_sim_df = pd.DataFrame(
        item_sim,
        index=data_matrix.columns,
        columns=data_matrix.columns
    )

    # Similar movies
    similar_items = item_sim_df[movie_id].sort_values(ascending=False)[1:6]

    # Users from similar items
    similar_users = set(
        ratings[ratings['movie_id'].isin(similar_items.index)]['user_id']
    )

    # Remove existing viewers
    existing_users = set(
        ratings[ratings['movie_id'] == movie_id]['user_id']
    )

    target_users = similar_users - existing_users

    # Attach names
    return user_lookup[user_lookup['user_id'].isin(target_users)].head(top_n)

#### Content Based Recommendations

In [5]:
def content_based(ratings, movie_content, user_id, top_n=10):

    watched = ratings[ratings['user_id'] == user_id]['movie_id'].values

    user_movies = movie_content[
        movie_content['movie id'].isin(watched)
    ].drop(columns=['movie id', 'movie title'])

    profile = user_movies.mean().values.reshape(1, -1)

    all_movies = movie_content.drop(columns=['movie id', 'movie title'])

    sim = cosine_similarity(profile, all_movies)[0]

    temp = movie_content.copy()
    temp['similarity'] = sim

    rec = temp[~temp['movie id'].isin(watched)]

    return rec[['movie id', 'movie title', 'similarity']] \
        .sort_values(by='similarity', ascending=False).head(top_n)

#### Hybrid Recommendations

In [6]:
def hybrid_system(ratings, items, movie_content, user_lookup, user_id, movie_id):

    # Run all models
    user_movies = user_based(ratings, items, user_id)
    content_movies = content_based(ratings, movie_content, user_id)
    target_users = item_based_users(ratings, user_lookup, movie_id)

    return {
        "User-Based Movies": user_movies,
        "Content-Based Movies": content_movies,
        "Target Users for Movie": target_users
    }

In [7]:
result = hybrid_system(
    ratings,
    items,
    movie_content,
    user_lookup,
    user_id=54,
    movie_id=50   # Example movie
)

In [20]:
# 1. User-Based → Convert list to table
user_movies_df = pd.DataFrame({
    "Recommended Movies (User-Based)": result["User-Based Movies"]
})


# 2. Content-Based → Already DataFrame (just rename nicely)
content_df = result["Content-Based Movies"].rename(columns={
    "movie id": "Movie ID",
    "movie title": "Movie Title",
    "similarity": "Similarity Score"
})


# 3. Item-Based → Users table
users_df = result["Target Users for Movie"].rename(columns={
    "user_id": "User ID",
    "user_name": "User Name"
})


In [25]:
def style_table(df, title=None):

    styled = (
        df.style
        .set_caption(title)
        .set_table_styles([

            # Caption styling
            {
                "selector": "caption",
                "props": [
                    ("font-size", "18px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("margin-bottom", "10px")
                ]
            },

            # Header styling
            {
                "selector": "th",
                "props": [
                    ("background-color", "#5AA647"),
                    ("color", "white"),
                    ("text-align", "center"),
                    ("padding", "10px"),
                    ("border", "1px solid #ccc")
                ]
            },

            # Hover effect
            {
                "selector": "tbody tr:hover",
                "props": [
                    ("background-color", "#eef7ea")
                ]
            }

        ])
        # Alternate rows
        .apply(
            lambda x: [
                "background-color: #f9f9f9" if i % 2 else ""
                for i in range(len(x))
            ],
            axis=0
        )
        # Cell formatting
        .set_properties(**{
            'text-align': 'left',
            'padding': '8px',
            'border': '1px solid #ccc'
        })
    )

    return styled

In [26]:
styled_user_df = style_table(
    user_movies_df,
    "User-Based Movie Recommendations"
)

display(styled_user_df)

styled_users_df = style_table(
    users_df,
    "Item Based Recommendation"
)

display(styled_users_df)

styled_content_df = style_table(
    content_df,
    "Content-Based Movie Recommendations"
).format({"Similarity Score": "{:.2f}"})  # format numbers

display(styled_content_df)

,Recommended Movies (User-Based)
0,"Truth About Cats & Dogs, The (1996)"
1,"Cable Guy, The (1996)"
2,Phenomenon (1996)
3,Mars Attacks! (1996)
4,Grosse Pointe Blank (1997)
5,"English Patient, The (1996)"
6,Air Force One (1997)
7,"People vs. Larry Flynt, The (1996)"
8,Volcano (1997)
9,Grumpier Old Men (1995)


,User ID,User Name
16,122,User-122
25,38,User-38
31,225,User-225
35,181,User-181
44,242,User-242
49,81,User-81
59,138,User-138
62,223,User-223
64,243,User-243
80,90,User-90


,Movie ID,Movie Title,Similarity Score
1555,1556,Condition Red (1995),0.85
916,917,Mercury Rising (1998),0.85
1490,1491,Tough and Deadly (1995),0.85
243,244,Smilla's Sense of Snow (1997),0.85
1024,1025,Fire Down Below (1997),0.85
1558,1559,Hostile Intentions (1994),0.85
53,54,Outbreak (1995),0.85
27,28,Apollo 13 (1995),0.85
854,855,Diva (1981),0.81
1104,1105,Firestorm (1998),0.79
